In [1]:
import os
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy.nddata import Cutout2D
from scipy.ndimage import find_objects
import sep
from scipy.ndimage import gaussian_filter, zoom
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, Tuple
from tqdm import tqdm
import csv

In [2]:
jseg_file = 'mastDownload/HLSP/hlsp_jades_jwst_nircam_goods-s-deep_segmentation_v2.0_drz.fits'
jsci_file = 'mastDownload/HLSP/hlsp_jades_jwst_nircam_goods-s-deep_f115w_v2.0/hlsp_jades_jwst_nircam_goods-s-deep_f115w_v2.0_drz.fits'
csci_file = 'mastDownload/HLSP/hlsp_ceers_jwst_nircam_nircam5_f115w_dr0.6_i2d/hlsp_ceers_jwst_nircam_nircam5_f115w_dr0.6_i2d.fits'

In [3]:
def export_galaxy_cutouts_parallel(
    sci_file,
    seg_file=None,
    outdir="galaxy_cutouts",
    galaxy_ids=None,
    galaxy_id_range=None,
    padding=50,
    sep_thresh=1.5,
    max_flux_threshold=None,
    min_area=20,
    max_distance=None,
    oversample_psf=4,
    filter_name="f150w",
    n_threads=8,
    max_seg_size=1000,
    max_cutout_size=500
):
    """
    Multithreaded galaxy cutout exporter with WCS for JWST CEERS/JADES.
    Cutouts are limited in size to avoid hanging or huge arrays.
    """
    import os, csv
    import numpy as np
    from astropy.io import fits
    from astropy.wcs import WCS
    import sep
    from scipy.ndimage import find_objects, zoom
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from tqdm import tqdm
    import stpsf
    from threading import Lock

    os.makedirs(outdir, exist_ok=True)

    # --- Load FITS ---
    with fits.open(sci_file) as hdul:
        sci_data = hdul["SCI"].data.astype(np.float32)
        err_data = hdul["ERR"].data.astype(np.float32)
        hdr = hdul["SCI"].header
        wcs = WCS(hdr)
        zp = hdr.get("ZP", 27.17)
        if "BKGMASK" in hdul:
            segmap = hdul["BKGMASK"].data.astype(np.int32)
            mode = "CEERS"
        elif seg_file is not None:
            segmap = fits.getdata(seg_file).astype(np.int32)
            mode = "JADES"
        else:
            raise ValueError("No segmentation map found.")

    # --- Metadata CSV ---
    metadata_path = os.path.join(outdir, "galaxy_metadata.csv")
    with open(metadata_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["zp", "mode"])
        writer.writeheader()
        writer.writerow({"zp": zp, "mode": mode})

    # --- PSF generation / caching ---
    filter_name = filter_name.upper()
    psf_dir = os.path.join(outdir, "psf")
    os.makedirs(psf_dir, exist_ok=True)
    psf_file = os.path.join(psf_dir, f"psf_{filter_name.lower()}.fits")
    if os.path.exists(psf_file):
        with fits.open(psf_file) as hdul:
            psf = hdul[0].data.astype(np.float32)
    else:
        psf_gen = stpsf.NIRCam()
        psf_gen.filter = filter_name
        psf_hdul = psf_gen.calc_psf(oversample=oversample_psf)
        psf_full = psf_hdul[0].data.astype(np.float32)
        psf = zoom(psf_full, 1/oversample_psf, order=3) if oversample_psf > 1 else psf_full.copy()
        psf /= psf.sum()
        fits.PrimaryHDU(data=psf).writeto(psf_file, overwrite=True)

    # --- Determine galaxy IDs ---
    if mode == "CEERS":
        # SEP detection on BKGMASK
        data_for_sep = sci_data.copy()
        data_for_sep[segmap==0] = 0
        bkg = sep.Background(data_for_sep)
        data_sub = data_for_sep - bkg
        objects = sep.extract(data_sub, thresh=sep_thresh)
        selected_ids = np.arange(1,len(objects)+1)
        slices = None
    else:
        max_id = segmap.max()
        all_ids = np.arange(1,max_id+1)
        if galaxy_ids is not None:
            selected_ids = [gid for gid in all_ids if gid in galaxy_ids]
        elif galaxy_id_range is not None:
            min_id,max_id_range = galaxy_id_range
            selected_ids = [gid for gid in all_ids if min_id <= gid <= max_id_range]
        else:
            selected_ids = all_ids
        slices = find_objects(segmap)
        objects = None

    galaxy_centers = {}
    lock = Lock()

    # --- Per-galaxy processing ---
    def _process_galaxy(gal_id):
        if mode == "CEERS":
            # --- CEERS branch (same as before) ---
            obj = objects[gal_id-1]
            x_center, y_center = obj["x"], obj["y"]

            x_min = int(max(x_center - padding, 0))
            x_max = int(min(x_center + padding, sci_data.shape[1]-1))
            y_min = int(max(y_center - padding, 0))
            y_max = int(min(y_center + padding, sci_data.shape[0]-1))

            sci_cut = sci_data[y_min:y_max+1, x_min:x_max+1].astype(np.float32)
            err_cut = err_data[y_min:y_max+1, x_min:x_max+1].astype(np.float32)

            # SEP segmentation
            bkg_cut = sep.Background(sci_cut)
            data_sub_cut = sci_cut - bkg_cut
            objects_local, segmap_cut = sep.extract(data_sub_cut, thresh=sep_thresh,
                                                    err=np.ascontiguousarray(err_cut, dtype=np.float32),
                                                    segmentation_map=True)
            if segmap_cut is None or len(objects_local)==0:
                return

            central_flux = sci_cut[segmap_cut==1].max()
            if max_flux_threshold is not None and central_flux < max_flux_threshold:
                return

            # Neighbor mask
            mask_sep = np.zeros_like(segmap_cut, dtype=np.uint8)
            x_center_local = sci_cut.shape[1]/2
            y_center_local = sci_cut.shape[0]/2
            distances = (objects_local["x"] - x_center_local)**2 + (objects_local["y"] - y_center_local)**2
            central_idx = np.argmin(distances)
            central_label = central_idx + 1
            for i_obj,obj_local in enumerate(objects_local):
                label = i_obj+1
                if label==central_label:
                    continue
                dx = obj_local["x"] - x_center_local
                dy = obj_local["y"] - y_center_local
                dist2 = dx**2 + dy**2
                area = obj_local["a"]*obj_local["b"]*np.pi
                if (max_distance is not None and np.sqrt(dist2) > max_distance) or (area < min_area):
                    continue
                mask_sep[segmap_cut==label] = 1
            mask_sep = (mask_sep>0).astype(np.uint8)

            x_cent = x_center
            y_cent = y_center

        else:  # JADES branch
            sl = slices[gal_id-1]
            if sl is None:
                return
            y_slice, x_slice = sl

            if (y_slice.stop - y_slice.start) > max_seg_size or (x_slice.stop - x_slice.start) > max_seg_size:
                y_center = (y_slice.start + y_slice.stop) // 2
                x_center = (x_slice.start + x_slice.stop) // 2
                y_start = max(y_center - max_seg_size//2, 0)
                x_start = max(x_center - max_seg_size//2, 0)
                y_end = min(y_start + max_seg_size, sci_data.shape[0])
                x_end = min(x_start + max_seg_size, sci_data.shape[1])
                y_slice = slice(y_start, y_end)
                x_slice = slice(x_start, x_end)

            mask = (segmap[y_slice, x_slice]==gal_id)
            if not mask.any():
                return
            galaxy_max_flux = sci_data[y_slice,x_slice][mask].max()
            if max_flux_threshold is not None and galaxy_max_flux < max_flux_threshold:
                return

            y_inds, x_inds = np.where(mask)
            x_min = max(x_inds.min()+x_slice.start-padding, 0)
            x_max = min(x_inds.max()+x_slice.start+padding, sci_data.shape[1]-1)
            y_min = max(y_inds.min()+y_slice.start-padding, 0)
            y_max = min(y_inds.max()+y_slice.start+padding, sci_data.shape[0]-1)

            sci_cut = sci_data[y_min:y_max+1, x_min:x_max+1].astype(np.float32)
            err_cut = err_data[y_min:y_max+1, x_min:x_max+1].astype(np.float32)

            # --- SEP segmentation for mask ---
            bkg_cut = sep.Background(sci_cut)
            data_sub_cut = sci_cut - bkg_cut
            objects_local, segmap_cut_tmp = sep.extract(data_sub_cut, thresh=sep_thresh,
                                                        err=np.ascontiguousarray(err_cut, dtype=np.float32),
                                                        segmentation_map=True)

            # Central galaxy segmentation
            segmap_cut = (segmap[y_min:y_max+1, x_min:x_max+1]==gal_id).astype(np.int16)

            # Neighbor mask
            mask_sep = np.zeros_like(segmap_cut, dtype=np.uint8)
            if objects_local is not None and len(objects_local) > 0:
                x_center_local = sci_cut.shape[1]/2
                y_center_local = sci_cut.shape[0]/2
                distances = (objects_local["x"] - x_center_local)**2 + (objects_local["y"] - y_center_local)**2
                central_idx = np.argmin(distances)
                central_label = central_idx + 1
                for i_obj,obj_local in enumerate(objects_local):
                    label = i_obj + 1
                    if label == central_label:
                        continue
                    dx = obj_local["x"] - x_center_local
                    dy = obj_local["y"] - y_center_local
                    dist2 = dx**2 + dy**2
                    area = obj_local["a"]*obj_local["b"]*np.pi
                    if (max_distance is not None and np.sqrt(dist2) > max_distance) or (area < min_area):
                        continue
                    mask_sep[segmap_cut_tmp == label] = 1
            mask_sep = (mask_sep>0).astype(np.uint8)

            x_cent = x_inds.mean()+x_slice.start
            y_cent = y_inds.mean()+y_slice.start

        # --- Apply hard cutout size ---
        if (x_max-x_min+1) > max_cutout_size:
            center = (x_min+x_max)//2
            x_min = max(center - max_cutout_size//2,0)
            x_max = x_min + max_cutout_size -1
        if (y_max-y_min+1) > max_cutout_size:
            center = (y_min+y_max)//2
            y_min = max(center - max_cutout_size//2,0)
            y_max = y_min + max_cutout_size -1

        # --- WCS ---
        cutout_wcs = wcs.deepcopy()
        cutout_wcs.wcs.crpix[0] -= x_min
        cutout_wcs.wcs.crpix[1] -= y_min

        # --- Save centers ---
        with lock:
            galaxy_centers[gal_id] = {"x_pix": float(x_cent), "y_pix": float(y_cent)}

        # --- Save cutouts ---
        fits.PrimaryHDU(sci_cut, header=cutout_wcs.to_header()).writeto(os.path.join(outdir,f"cutout_sci_{gal_id}.fits"),overwrite=True)
        fits.PrimaryHDU(err_cut, header=cutout_wcs.to_header()).writeto(os.path.join(outdir,f"cutout_err_{gal_id}.fits"),overwrite=True)
        fits.PrimaryHDU(segmap_cut.astype(np.int16), header=cutout_wcs.to_header()).writeto(os.path.join(outdir,f"segmap_{gal_id}.fits"),overwrite=True)
        fits.PrimaryHDU(mask_sep, header=cutout_wcs.to_header()).writeto(os.path.join(outdir,f"mask_{gal_id}.fits"),overwrite=True)

    # --- Run multithreaded ---
    with ThreadPoolExecutor(max_workers=n_threads) as executor:
        futures = [executor.submit(_process_galaxy, gid) for gid in selected_ids]
        for _ in tqdm(as_completed(futures), total=len(futures), desc="Processing galaxies"):
            pass

    # --- Save centers ---
    centers_path = os.path.join(outdir,"galaxy_centers.csv")
    with open(centers_path,"w",newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["gal_id","x_pix","y_pix"])
        writer.writeheader()
        for gid in sorted(galaxy_centers.keys()):
            writer.writerow({"gal_id":gid,**galaxy_centers[gid]})

    return psf, zp, mode, galaxy_centers


In [4]:
f1 = export_galaxy_cutouts_parallel(jsci_file, jseg_file, filter_name='f115w', outdir="galaxy_cutouts", max_flux_threshold=0.1, galaxy_id_range=(202206,211158))

Set DATE-AVG to '2022-10-08T02:26:24.819' from MJD-AVG.
Set DATE-END to '2022-10-09T02:34:11.369' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    22.090597 from OBSGEO-[XYZ].
Set OBSGEO-H to 1234812278.786 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Processing galaxies: 100%|████████████████████████████████████████████████████████| 8953/8953 [00:12<00:00, 740.29it/s]


In [5]:
f2 = export_galaxy_cutouts_parallel(csci_file, None, filter_name='f115w', outdir="ceers_galaxy_cutouts", max_flux_threshold=0.1, galaxy_id_range=(0,1090))

Set DATE-AVG to '2022-12-23T09:54:11.188' from MJD-AVG.
Set DATE-END to '2022-12-24T19:19:42.661' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to     8.534125 from OBSGEO-[XYZ].
Set OBSGEO-H to 1680304337.829 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Processing galaxies: 100%|██████████████████████████████████████████████████████████| 138/138 [00:01<00:00, 127.04it/s]
